# Brief overview of algorithm
Our approach is split into 2 parts, creating a graph, and then running dijkstra's on the graph.
# Creating a graph:
Idea of graph - each node represents a stop and edges between nodes represent a direct trip between stops. The graph is directed.
For each stop, determine and sort the next possible departure times so that it is easy to look-up connection times.
Iterate through each trip, and calculate the travel time, and wait time store, these in the edges.
- Travel time: The difference between the departure time at the first stop to the arrival time of the next stop. 
- Wait Time: The time until the next trip departs from the stop, taking into account edge cases where the next trip may be on the next day. 

# Dijkstra's algorithm:
for each node you visit, consider all outgoing edges and calculate the new arrival time at the next node by summing the current node's arrival time, wait time and travel time.
Update the shortest travel time to each node if a faster route is found. 

In [1]:
import pandas as pd
import plotly.express as px
import heapq
from datetime import timedelta
from geopy.distance import geodesic
from itertools import combinations
import os
from pyhive import hive
import warnings
import re
import ast
import plotly.graph_objects as go

from typing import Dict, List, Tuple, TypedDict
from pandas import DataFrame

stop_times_df = pd.read_csv("data/stop_times_df.csv")
stops_df = pd.read_csv("data/stops_df.csv")
stop_to_stop_df = pd.read_csv("data/stop_to_stop_df.csv")
route_desc_df = pd.read_csv("data/route_desc.csv")
routes_df = pd.read_csv("data/routes.csv")

In [2]:
def parse_time(time_str: str = '') -> timedelta:
    """Convert time strings to timedelta"""
    hours, minutes, seconds = map(int, time_str.split(':'))
    return timedelta(hours=hours % 24, minutes=minutes, seconds=seconds)


stop_times_df['arrival_time'] = stop_times_df['arrival_time'].apply(parse_time)
stop_times_df['departure_time'] = stop_times_df['departure_time'].apply(parse_time)

In [3]:
def calculate_walking_time(distance_meters: float = 0.0) -> timedelta:
    """
    Calculate the walking timedelta of a given distance.
    We assumed that a normal person can walk 50 meters in 1 minute.
    Otherwise, we assumed that the walking timedelta is 2 minutes
    """
    return max(timedelta(minutes=distance_meters / 50), timedelta(minutes=2))

In [4]:
def build_graph(stop_times_df: DataFrame = None,
                stops_df: DataFrame = None,
                stop_to_stop_df: DataFrame = None
                ) -> Dict[str, List[Tuple[str, timedelta, timedelta, timedelta, str]]]:
    graph = {stop: [] for stop in stops_df['stop_id']}
    next_departures = {}

    # Process stop times for public transport connections
    for stop_id, group in stop_times_df.groupby('stop_id'):
        sorted_group = group.sort_values('departure_time')
        next_departures[stop_id] = {row['trip_id']: row['departure_time'] for _, row in sorted_group.iterrows()}

    for _, group in stop_times_df.groupby('trip_id'):
        sorted_group = group.sort_values('departure_time')
        for i in range(len(sorted_group) - 1):
            row_current = sorted_group.iloc[i]
            row_next = sorted_group.iloc[i + 1]
            stop_a = row_current['stop_id']
            stop_b = row_next['stop_id']
            trip_id = row_current['trip_id']
            arrival_time = row_current['arrival_time']
            departure_time = row_next['departure_time']
            travel_time = departure_time - arrival_time
            wait_time = (next_departures.get(stop_b, {}).get(trip_id, departure_time) - arrival_time)

            if wait_time < timedelta(0):
                wait_time += timedelta(days=1)

            graph[stop_a].append((stop_b, travel_time, wait_time, departure_time, trip_id))

    # Add walking connections without a trip ID (or use None)
    for _, row in stop_to_stop_df.iterrows():
        stop_a = row['stop_id_a']
        stop_b = row['stop_id_b']
        distance = row['distance']
        walking_time = calculate_walking_time(distance)
        graph[stop_a].append((stop_b, walking_time, timedelta(0), None, None))
        graph[stop_b].append((stop_a, walking_time, timedelta(0), None, None))

    return graph


In [5]:
# Create the graph instance
graph = build_graph(stop_times_df, stops_df, stop_to_stop_df)

In [6]:
def get_probability_of_train_being_on_time(start_stop_id: str = '',
                                           end_stop_id: str = '',
                                           schedule_departure_time: timedelta = None,
                                           schedule_arrival_time: timedelta = None,
                                           transport_type: str = '',
                                           date: str = '',
                                           wait_time: float = 0.0) -> float:
    """In Switzerland, TRAINS ARE ALWAYS ON TIME ;)"""
    return 100.0

In [7]:
def dijkstra_time_aware(graph: Dict[str, List[Tuple[str, timedelta, timedelta, timedelta, str]]] = {},
                        start_id: str = '', end_id: str = '', start_time: str = '',
                        # threshold: float = 99.0
                        threshold: float = 0.0
                        ) -> List[Tuple[str, float, str]]:
    pq = [(start_time, start_id, [], None)]
    visited = {}

    while pq:
        current_time, current_stop, path, last_trip_id = heapq.heappop(pq)

        if current_stop == end_id:
            return [(stop, (time - start_time).total_seconds() / 60, trip_id if trip_id else "walking")
                    for stop, time, trip_id in path]

        if current_stop in visited and visited[current_stop] <= current_time:
            continue
        visited[current_stop] = current_time

        for next_stop, travel_time, _, departure_time, next_trip_id in graph[current_stop]:
            if departure_time is None:
                departure_time_adjusted = current_time
                current_trip_id = "walking"
                probability = 100.0
            else:
                if current_time > departure_time:
                    continue
                departure_time_adjusted = max(departure_time, current_time)
                current_trip_id = next_trip_id

                # Get probability of train on time
                probability = get_probability_of_train_being_on_time(current_stop, next_stop,
                                                                     departure_time, departure_time + travel_time,
                                                                     "Underground", "2024-01-01", 2)

            if probability > threshold:

                if next_trip_id is not None and last_trip_id != next_trip_id:
                    departure_time_adjusted = max(departure_time_adjusted, current_time + timedelta(minutes=2))

                new_time = departure_time_adjusted + travel_time
                new_path = path + [(next_stop, new_time, current_trip_id)]

                if next_stop not in visited or new_time < visited[next_stop]:
                    heapq.heappush(pq, (new_time, next_stop, new_path, next_trip_id))

    return []

In [8]:
# Initial stop and time
#start_stop_id = '8501214'
#start_time = timedelta(hours=0)
#end_stop_id = '8588158'

In [9]:
def parse_input(input: DataFrame = None) -> Dict:
    """
    Parses the input from the GUI to a Dictionary with the desired Keys for further processing.
    
    Args:
        input: Dataframe of one entry with the initial and destination stops ids, the starting time, and the confidence of the trip 
    Returns:
        Dictionary with the desired Keys:
        
            'start_stop_id': (str) start_stop_id
            'start_time': (timedelta) parsed_time
            'end_stop_id': (str) end_stop_id
            'confidence': (float) confidence        
     """
    start_stop_id = stops_df.loc[stops_df['stop_name'] == input['from'], 'stop_id'].iloc[0]
    end_stop_id = stops_df.loc[stops_df['stop_name'] == input['to'], 'stop_id'].iloc[0]
    confidence = input['confidence']

    if input['time']:
        parsed_time = timedelta(hours=input['time'].hour, minutes=input['time'].minute)
    else:
        parsed_time = timedelta(hours=0)

    parsed_input = {
        'start_stop_id': start_stop_id,
        'start_time': parsed_time,
        'end_stop_id': end_stop_id,
        'confidence': confidence
    }
    return parsed_input

In [10]:
def extract_route_id(trip_id: str = '') -> str:
    """Check if it is a standard route_id and return it or None"""
    match = re.search(r'TA\.(\d+-[A-Za-z0-9-]+-j24-\d+)', trip_id)
    if match:
        return match.group(1)
    return None

In [11]:
def parse_output(graph: Dict = {},
                 start_id: str = '', end_id: str = '', start_time: str = '',
                 routes_df: DataFrame = None, route_desc_df: DataFrame = None, stops_df: DataFrame = None,
                 threshold: float = 0.0
                 ) -> DataFrame:
    """
    TODO: Comment the method properly with args and returns
    This method is for aesthetics purposes
    """
    path = dijkstra_time_aware(graph, start_id, end_id, start_time, threshold)
    journey_details = []

    print("\nJourney Details:\n")

    if path:
        first_stop, first_time, first_trip_id = path[0]
        first_time_delta = timedelta(minutes=first_time)
        first_adjusted_time = (start_time + first_time_delta)

        first_stop_info = stops_df[stops_df['stop_id'] == start_id]
        if not first_stop_info.empty:
            first_stop_name = first_stop_info['stop_name'].values[0]
            first_stop_lat = first_stop_info['stop_lat'].values[0]
            first_stop_lon = first_stop_info['stop_lon'].values[0]
        else:
            first_stop_name, first_stop_lat, first_stop_lon = "Unknown stop", None, None

        if first_trip_id == "walking":
            first_transport_mode = "Walk"
            first_adjusted_time = start_time
        else:
            first_route_id = extract_route_id(first_trip_id)
            first_route_type = routes_df[routes_df['route_id'] == first_route_id]['route_type'].iloc[0] if not \
                routes_df[routes_df['route_id'] == first_route_id].empty else None
            first_transport_mode = route_desc_df[route_desc_df['route_type'] == first_route_type]['EN'].iloc[
                0] if first_route_type and not route_desc_df[
                route_desc_df['route_type'] == first_route_type].empty else "Unknown mode"

        journey_details.append({
            "stop_id": start_id,
            "stop_name": first_stop_name,
            "stop_lat": first_stop_lat,
            "stop_lon": first_stop_lon,
            "time_at_stop": first_adjusted_time,
            "form_of_transportation": first_transport_mode
        })

        print(f"{first_transport_mode} from Starting Location: {first_stop_name}, Time at stop: {first_adjusted_time}")

    for index, (stop, time, trip_id) in enumerate(path[1:], start=1):
        current_time = timedelta(minutes=time)
        adjusted_time = (start_time + current_time)
        stop_info = stops_df[stops_df['stop_id'] == stop]

        if not stop_info.empty:
            stop_name = stop_info['stop_name'].values[0]
            stop_lat = stop_info['stop_lat'].values[0]
            stop_lon = stop_info['stop_lon'].values[0]
        else:
            stop_name, stop_lat, stop_lon = "Unknown stop", None, None

        if trip_id == "walking":
            transport_mode = "Walk"
        else:
            route_id = extract_route_id(trip_id)

            route_type = None
            if not routes_df[routes_df['route_id'] == route_id].empty:
                route_type = routes_df[routes_df['route_id'] == route_id]['route_type'].iloc[0]

            transport_mode = "Unknown mode"
            if route_type and not route_desc_df[route_desc_df['route_type'] == route_type].empty:
                transport_mode = route_desc_df[route_desc_df['route_type'] == route_type]['EN'].iloc[0]

        print(f"{transport_mode} to {stop_name}, Time at stop: {adjusted_time}")

        journey_details.append({
            "stop_id": stop,
            "stop_name": stop_name,
            "stop_lat": stop_lat,
            "stop_lon": stop_lon,
            "time_at_stop": adjusted_time,
            "form_of_transportation": transport_mode
        })

    journey_details_df = pd.DataFrame(journey_details)
    return journey_details_df

In [12]:
def plot_route(df: DataFrame = None):
    """
    Create a color map for unique forms of transportation
    """
    unique_transportation = df['form_of_transportation'].unique()
    color_map = {transport: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
                 for i, transport in enumerate(unique_transportation)}

    # Create the figure
    fig = go.Figure()

    # Track which transport forms have been added to the legend
    transports_added = {}

    # Iterate over the DataFrame and draw lines and markers between each row
    for i in range(len(df)):
        transport_type = df.iloc[i]['form_of_transportation']
        show_legend = not transports_added.get(transport_type, False)

        if i < len(df) - 1:
            # Add lines connecting the stops
            fig.add_trace(go.Scattermapbox(
                mode="lines",
                lon=[df.iloc[i]['stop_lon'], df.iloc[i + 1]['stop_lon']],
                lat=[df.iloc[i]['stop_lat'], df.iloc[i + 1]['stop_lat']],
                line={'width': 2, 'color': color_map[transport_type]},
                name=transport_type,
                showlegend=show_legend
            ))

        # Add markers for each stop
        fig.add_trace(go.Scattermapbox(
            mode="markers+text",
            lon=[df.iloc[i]['stop_lon']],
            lat=[df.iloc[i]['stop_lat']],
            text=[df.iloc[i]['time_at_stop'].total_seconds() // 3600],  # Display time in hours
            textposition="bottom right",
            marker={'size': 10, 'color': color_map[transport_type]},
            name=transport_type,
            showlegend=False
        ))

        # Mark this transport form as added to the legend
        transports_added[transport_type] = True

    # Update the layout
    fig.update_layout(
        margin={'l': 0, 't': 0, 'b': 0, 'r': 0},
        mapbox={
            'style': "open-street-map",
            'center': {'lon': df['stop_lon'].mean(), 'lat': df['stop_lat'].mean()},
            'zoom': 10  # Adjust zoom level as needed
        }
    )

    fig.show()

In [13]:
# Initial stop and time

# You need at a minimum to allow us to enter a starting point an end point, a maximum expected time of arrival, and a minimum confidence of arriving before the maximum expected time.
# Optionally, if you support any day of the week (Monday to Sunday), you should allow us to pick one of the supported day. Otherwise, be specific of which day of the week the journey is valid so that we only compare with SBB's journey for that day.
# Do not specify a week of year (just use a recent timetable, e.g. 2024.5.16 which is available from the com490final db).
# It must be interactive, so that we can try different routes / time of day.

# Inputs: 
# - starting point
# - endpoint
# - (expected time of arrival)
# - a minimum confidence of arriving before the maximum expected time
# - Date 

# Outputs (after algorithm):
# The trajectory (Flon -> Montelly with metro -> walk to Malley -> ...)
# Can be done also via map and with ID's and so on..

import ipywidgets as widgets
from IPython.display import display

global stops_df

#print(stops_df)
stop_list = sorted(stops_df['stop_name'].tolist())

# Create the first dropdown widget
from_dropdown = widgets.Dropdown(
    options=stop_list,
    value=stop_list[0],
    description='From:',
    disabled=False
)

# Create the second dropdown widget
to_dropdown = widgets.Dropdown(
    options=stop_list,
    value=stop_list[100],
    description='To:',
    disabled=False
)

## We might want to include day as well!
# Create a time selector widget
time_picker = widgets.TimePicker(
    description='Departure:',
    disabled=False
)

# Create a float slider widget
float_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=100.0,
    step=0.1,
    description='Confidence:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Create a button widget
button = widgets.Button(
    description='Find Route',
    disabled=False,
    button_style='danger',  # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='check'  # (FontAwesome names without the `fa-` prefix)
)

# Create an output area
output = widgets.Output()


def on_button_clicked(b):
    """Behaviour (callback function) for the button click event"""
    global start_stop_id, start_time, end_stop_id

    with output:
        output.clear_output()
        input = {
            'from': from_dropdown.value,
            'to': to_dropdown.value,
            'time': time_picker.value,
            'confidence': float_slider.value
        }
        parsed_input = parse_input(input)

        start_stop_id = parsed_input['start_stop_id']
        start_time = parsed_input['start_time']
        end_stop_id = parsed_input['end_stop_id']

        df = parse_output(graph, start_stop_id, end_stop_id, start_time, routes_df, route_desc_df, stops_df,
                          threshold=parsed_input['confidence'])
        plot_route(df)


# Attach the callback function to the button
button.on_click(on_button_clicked)

# Display the widgets
display(from_dropdown, to_dropdown, time_picker, float_slider, button, output)

# You need at a minimum to show the details of the public transport journey (including walking if any). 
# It must be presented in a way that can be easily compared with SBB, e.g. list of human-readable stop names
# and arrival departures, or list of public transport line ID or numbers that can be easily compared with 
# SBB, or map information showing the stops on a map and different transports (e.g. using different colors).


Dropdown(description='From:', options=('Bel-Air LEB', 'Bel-Air LEB', 'Belmont-sur-L., Blessoney', 'Belmont-sur…

Dropdown(description='To:', index=100, options=('Bel-Air LEB', 'Bel-Air LEB', 'Belmont-sur-L., Blessoney', 'Be…

TimePicker(value=None, description='Departure:', step=60.0)

FloatSlider(value=0.0, continuous_update=False, description='Confidence:', readout_format='.1f')

Button(button_style='danger', description='Find Route', icon='check', style=ButtonStyle(), tooltip='Click me')

Output()

In [63]:
df = parse_output(graph, start_stop_id, end_stop_id, start_time, routes_df, route_desc_df, stops_df, threshold=0.0)


Journey Details:

Walk from Starting Location: Belmont-sur-L., Chaffeises, Time at stop: 0:00:00
Walk to Pully, Reymondin, Time at stop: 0:16:21.737484
Walk to Pully, Vignes, Time at stop: 0:20:52.154052
Bus to Pully, port, Time at stop: 0:22:52.154052
Walk to Pully, Somais, Time at stop: 0:28:53.543088
Bus to Pully, Châtaignier, Time at stop: 0:32:00
Walk to Pully, Verney, Time at stop: 0:38:15.638016
Walk to Pully, Bourdonnière, Time at stop: 0:43:04.518036


In [64]:
df

,stop_id,stop_name,stop_lat,stop_lon,time_at_stop,form_of_transportation
0,8593795,"Belmont-sur-L., Chaffeises",46.514120,6.673602,0 days 00:00:00,Walk
1,8592195,"Pully, Reymondin",46.509243,6.665625,0 days 00:16:21.737484,Walk
2,8592199,"Pully, Vignes",46.507217,6.665508,0 days 00:20:52.154052,Walk
3,8588439,"Pully, port",46.506264,6.661403,0 days 00:22:52.154052,Bus
4,8588440,"Pully, Somais",46.506702,6.657531,0 days 00:28:53.543088,Walk
5,8588441,"Pully, Châtaignier",46.506462,6.653803,0 days 00:32:00,Bus
6,8592198,"Pully, Verney",46.506470,6.649725,0 days 00:38:15.638016,Walk
7,8592170,"Pully, Bourdonnière",46.506530,6.646590,0 days 00:43:04.518036,Walk
